**Google Colab / NVIDIA T4 · Unsloth + Hugging Face TRL**

Goal: Teach a pretrained Qwen2.5-1.5B language model to respond to technician notes using a consistent, cautious maintenance-report format.

This notebook builds two experimental branches (run one at a time):
- A — Direct SFT: original Qwen base → LoRA + instruction SFT.
- B — Sequential SFT: original Qwen + yesterday's Stage 1 Qwen LoRA adapter → additional instruction SFT.
Why two branches? We want to test whether non-instruction domain adaptation helps beyond direct instruction fine-tuning. Separate checkpoints are essential to answer that question.

Pipeline: installation → Drive paths → inspect SFT data → group-aware validation split → prompt/completion formatting → load base + correct adapter → completion-only loss → train → validate → save LoRA adapter → generation test

In [1]:

%pip install -q -U unsloth datasets trl peft accelerate bitsandbytes pandas

**`Step 1: Import libraries and GPU`**

In [2]:
import unsloth
import os, gc, json, time, random, re, hashlib
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

assert torch.cuda.is_available(), "Colab: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
PyTorch: 2.11.0+cu130


**Step 2: Folder Paths and Drive Mount**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/llm-post-training-experimental-study')
SFT_FILE = PROJECT_ROOT / 'data/processed/sft_prepared.jsonl'
STAGE1_ADAPTER = PROJECT_ROOT / 'models/stage1_non_instruction_adapter'
MODEL_DIR = PROJECT_ROOT / 'models'
METRIC_DIR = PROJECT_ROOT / 'results/metrics'
LOCAL_LOGS = Path('/content/stage2_sft_logs')
for p in [MODEL_DIR, METRIC_DIR, LOCAL_LOGS]: p.mkdir(parents=True, exist_ok=True)

assert SFT_FILE.exists(), f"Missing SFT dataset: {SFT_FILE}"
print('SFT data:', SFT_FILE)
print('Stage 1 adapter present:', (STAGE1_ADAPTER / 'adapter_config.json').exists())


Mounted at /content/drive
SFT data: /content/drive/MyDrive/llm-post-training-experimental-study/data/processed/sft_prepared.jsonl
Stage 1 adapter present: True


**Step 3: Config**

In [4]:
BRANCH = 'B'  # 'A' = base Qwen + direct SFT; 'B' = Stage 1 adapted Qwen + SFT
BASE_MODEL_NAME = 'unsloth/Qwen2.5-1.5B-bnb-4bit'
MAX_SEQ_LENGTH = 768
SEED = 42
LORA_R = 16
LORA_ALPHA = 32
BATCH_SIZE = 1
ACCUMULATION_STEPS = 4
MAX_STEPS = 40
LEARNING_RATE = 1e-4
WARMUP_STEPS = 5

assert BRANCH in {'A','B'}
random.seed(SEED)
torch.manual_seed(SEED)
OUT_ADAPTER = MODEL_DIR / ('stage2_sft_only_adapter' if BRANCH == 'A' else 'stage2_domain_sft_adapter')
OUT_METRICS = METRIC_DIR / f'stage2_sft_branch_{BRANCH.lower()}_metrics.json'
print('Branch:', BRANCH, '→', OUT_ADAPTER)


Branch: B → /content/drive/MyDrive/llm-post-training-experimental-study/models/stage2_domain_sft_adapter


**STep 4: Dataset load**

In [8]:
from datasets import load_dataset


sft_raw = load_dataset('json', data_files=str(SFT_FILE), split='train')
assert {'prompt', 'completion'} <= set(sft_raw.column_names), sft_raw.column_names
assert len(sft_raw) == 180, f'Expected 180 records from our dataset, got {len(sft_raw)}.'
assert all(isinstance(x, str) and x.strip() for x in sft_raw['prompt'])
assert all(isinstance(x, str) and x.strip() for x in sft_raw['completion'])
assert all(p.rstrip().endswith('### Response:') for p in sft_raw['prompt']), 'Unexpected prompt format: inspect Notebook 01 preprocessing.'

# The remaining two columns are optional consistency checks.
if 'text' in sft_raw.column_names:
    assert all(row['text'] == row['prompt'] + row['completion'] for row in sft_raw), 'Some combined text fields do not match prompt + completion.'
if 'output' in sft_raw.column_names:
    assert all(row['output'] == row['completion'] for row in sft_raw), 'Some output fields differ from completion.'

print('Records:', len(sft_raw), '| Columns:', sft_raw.column_names)
print('Exact duplicate prompts:', len(sft_raw) - len(set(sft_raw['prompt'])))
print('First prompt:\n', sft_raw[0]['prompt'][:700])
print('First completion:\n', sft_raw[0]['completion'][:1000])


Records: 180 | Columns: ['prompt', 'completion', 'text', 'output']
Exact duplicate prompts: 0
First prompt:
 ### Instruction:
Transform this technician note into a structured industrial maintenance report. Distinguish observations from unverified causes, and do not invent facts.

Technician note: P-204: Vibration rose across two shifts; speed remained steady.

### Response:

First completion:
 Equipment: P-204 (centrifugal pump)
Reported issue: Vibration increasing near the drive-end bearing.
Observed evidence: Vibration rose across two shifts; speed remained steady.
Possible contributors (not confirmed): alignment, bearing condition, or structural looseness.
Missing information: verified vibration measurements, operating load, alignment history.
Recommended follow-up: Log vibration trend; request condition-monitoring review under approved procedures.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.


**Step 5: Inspect before training**

In [9]:
for i in [0, 1, 2, 30, 90, 150]:
    print('\n' + '='*70)
    print('ROW', i)
    print('PROMPT:', sft_raw[i]['prompt'][:550])
    print('COMPLETION:', sft_raw[i]['completion'][:900])
print('\nReview these for unsupported observations and diagnoses before training.')




ROW 0
PROMPT: ### Instruction:
Transform this technician note into a structured industrial maintenance report. Distinguish observations from unverified causes, and do not invent facts.

Technician note: P-204: Vibration rose across two shifts; speed remained steady.

### Response:

COMPLETION: Equipment: P-204 (centrifugal pump)
Reported issue: Vibration increasing near the drive-end bearing.
Observed evidence: Vibration rose across two shifts; speed remained steady.
Possible contributors (not confirmed): alignment, bearing condition, or structural looseness.
Missing information: verified vibration measurements, operating load, alignment history.
Recommended follow-up: Log vibration trend; request condition-monitoring review under approved procedures.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.

ROW 1
PROMPT: ### Instruction:
Prepare a professional maintenance handover report based only on the field note below. Identify informat

**Step 6: Split by maintainence scenario**

In [10]:
assert len(sft_raw) % 3 == 0
# Verify by inspection above that rows 0-2 represent the same underlying scenario.
groups = list(range(len(sft_raw)//3))
random.Random(SEED).shuffle(groups)
n_val_groups = max(1, round(0.15 * len(groups)))
val_groups = set(groups[:n_val_groups])
train_ids = [i for i in range(len(sft_raw)) if (i//3) not in val_groups]
val_ids = [i for i in range(len(sft_raw)) if (i//3) in val_groups]
train_raw = sft_raw.select(train_ids)
val_raw = sft_raw.select(val_ids)
assert not ({i//3 for i in train_ids} & {i//3 for i in val_ids})
print('Train:', len(train_raw), 'Validation:', len(val_raw))


Train: 153 Validation: 27


In [11]:
# Keep the prompt and response exactly as prepared in Notebook 01.
train_data = train_raw.select_columns(['prompt', 'completion'])
val_data = val_raw.select_columns(['prompt', 'completion'])

print('TRAIN PROMPT:\n', train_data[0]['prompt'][:500])
print('TRAIN COMPLETION:\n', train_data[0]['completion'][:800])


TRAIN PROMPT:
 ### Instruction:
Transform this technician note into a structured industrial maintenance report. Distinguish observations from unverified causes, and do not invent facts.

Technician note: P-204: Vibration rose across two shifts; speed remained steady.

### Response:

TRAIN COMPLETION:
 Equipment: P-204 (centrifugal pump)
Reported issue: Vibration increasing near the drive-end bearing.
Observed evidence: Vibration rose across two shifts; speed remained steady.
Possible contributors (not confirmed): alignment, bearing condition, or structural looseness.
Missing information: verified vibration measurements, operating load, alignment history.
Recommended follow-up: Log vibration trend; request condition-monitoring review under approved procedures.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.


**Step 7: Load Model and weights**

In [12]:
from peft import PeftConfig

if BRANCH == 'B':
    assert (STAGE1_ADAPTER / 'adapter_config.json').exists(), f'No saved Stage 1 adapter at {STAGE1_ADAPTER}'
    prior = PeftConfig.from_pretrained(str(STAGE1_ADAPTER))
    prior_base = str(prior.base_model_name_or_path).lower()
    print('Saved adapter base:', prior.base_model_name_or_path)
    assert 'qwen2.5' in prior_base and '1.5b' in prior_base,         'Stage 1 folder appears to contain a different model (e.g., TinyLlama). STOP and correct its path.'
    load_from = str(STAGE1_ADAPTER)
else:
    load_from = BASE_MODEL_NAME

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=load_from,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Loaded:', load_from)
print('EOS:', repr(tokenizer.eos_token))


Saved adapter base: unsloth/Qwen2.5-1.5B-bnb-4bit
==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Loaded: /content/drive/MyDrive/llm-post-training-experimental-study/models/stage1_non_instruction_adapter
EOS: '<|endoftext|>'


**Step 8: for branch A (Pure Qwen Model, not merged), attach LORA**

In [13]:
if BRANCH == 'A':
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_dropout=0, bias='none', use_gradient_checkpointing='unsloth', random_state=SEED,
    )

FastLanguageModel.for_training(model)
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable > 0, 'No trainable adapter weights; reload Stage 1 adapter in trainable mode.'
print('Trainable parameters:', f'{trainable:,}')


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Trainable parameters: 18,464,768


**Step 9: Add "<eos>" **

In [14]:
def add_eos(row):
    c = row['completion']
    return {'completion': c if c.endswith(tokenizer.eos_token) else c + tokenizer.eos_token}

train_data = train_data.map(add_eos)
val_data = val_data.map(add_eos)

lengths = [len(tokenizer(r['prompt'] + r['completion'], add_special_tokens=False)['input_ids'])
           for r in list(train_data) + list(val_data)]
print('Token lengths min / median / max:', min(lengths), sorted(lengths)[len(lengths)//2], max(lengths))
print('Rows longer than max length:', sum(x > MAX_SEQ_LENGTH for x in lengths))
assert max(lengths) <= MAX_SEQ_LENGTH, 'Some records are too long. Adjust MAX_SEQ_LENGTH after inspecting them.'


Map:   0%|          | 0/153 [00:00<?, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Token lengths min / median / max: 139 152 168
Rows longer than max length: 0


**Step 10: Train model with SFTTrainer**

In [15]:
training_args = SFTConfig(
    output_dir=str(LOCAL_LOGS / f'branch_{BRANCH.lower()}'),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    completion_only_loss=True,
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim='adamw_8bit',
    seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=training_args,
)
print('Trainer ready. Confirm loss masking in the next cell before training.')


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/153 [00:00<?, ? examples/s]

Unsloth: Building labels for train dataset (num_proc=2):   0%|          | 0/153 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/27 [00:00<?, ? examples/s]

Unsloth: Building labels for eval dataset (num_proc=2):   0%|          | 0/27 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer ready. Confirm loss masking in the next cell before training.


**Step 11: Trainer loss**

In [16]:
batch = next(iter(trainer.get_train_dataloader()))
labels = batch['labels'][0]
non_ignored = (labels != -100).sum().item()
print('Batch input tokens:', int((batch['attention_mask'][0] != 0).sum()))
print('Tokens contributing to loss:', non_ignored)
assert non_ignored > 0, 'No response tokens are contributing to the loss!'
print('First learned target tokens:')
print(tokenizer.decode(labels[labels != -100][:100].tolist(), skip_special_tokens=False))


KeyError: 'attention_mask'

**Evalutation**

In [17]:
gc.collect(); torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()
train_seconds = round(time.perf_counter()-t0, 2)
peak_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
print('Training loss:', train_result.training_loss)
print('Training duration (sec):', train_seconds)
print('Peak allocated VRAM (GiB):', peak_gb)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 153 | Num Epochs = 2 | Total steps = 40
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
1,3.275070
2,2.964631
3,3.038845
4,2.936209
5,2.481402
6,2.371426
7,2.391906
8,1.787491
9,1.571680
10,1.660916


Training loss: 1.3927148193120957
Training duration (sec): 76.74
Peak allocated VRAM (GiB): 1.382


In [19]:
eval_metrics = trainer.evaluate()
print(json.dumps(eval_metrics, indent=2))


Filter:   0%|          | 0/27 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training Loss,Validation Loss,Step
0.784886,1.062393,40


{
  "eval_loss": 1.0623928308486938
}


**Save adapter and record**

In [20]:
OUT_ADAPTER.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUT_ADAPTER))
tokenizer.save_pretrained(str(OUT_ADAPTER))

run_record = {
    'branch': BRANCH, 'base_model': BASE_MODEL_NAME,
    'starting_checkpoint': str(STAGE1_ADAPTER) if BRANCH == 'B' else BASE_MODEL_NAME,
    'adapter_path': str(OUT_ADAPTER), 'dataset': str(SFT_FILE),
    'train_rows': len(train_data), 'validation_rows': len(val_data),
    'seed': SEED, 'max_length': MAX_SEQ_LENGTH, 'steps': MAX_STEPS,
    'lora_r_requested': LORA_R, 'lora_alpha_requested': LORA_ALPHA,
    'learning_rate': LEARNING_RATE, 'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': ACCUMULATION_STEPS,
    'train_loss': float(train_result.training_loss),
    'eval_loss': float(eval_metrics['eval_loss']),
    'training_seconds': train_seconds,
    'peak_gpu_memory_allocated_gib': peak_gb,
    'gpu': torch.cuda.get_device_name(0),
}
OUT_METRICS.write_text(json.dumps(run_record, indent=2), encoding='utf-8')
assert (OUT_ADAPTER / 'adapter_config.json').exists()
print('Saved adapter:', OUT_ADAPTER)
print('Saved metrics:', OUT_METRICS)


Saved adapter: /content/drive/MyDrive/llm-post-training-experimental-study/models/stage2_domain_sft_adapter
Saved metrics: /content/drive/MyDrive/llm-post-training-experimental-study/results/metrics/stage2_sft_branch_b_metrics.json


**Inference with same prompt**

In [21]:
FastLanguageModel.for_inference(model)
model.eval()
new_instruction = (
    'Transform this technician note into a structured industrial maintenance report. '
    'Distinguish observations from unverified causes, and do not invent facts.\n\n'
    'Technician note: Conveyor CV-118 stopped twice this morning. '
    'Operator reports an intermittent squeal near the drive housing; '
    'no temperature or vibration readings were recorded.'
)
# Same formatting as build_instruction_prompt() in Notebook 01.
inference_prompt = '### Instruction:\n' + new_instruction.strip() + '\n\n### Response:\n'
inputs = tokenizer(inference_prompt, return_tensors='pt').to('cuda')
with torch.inference_mode():
    generated = model.generate(
        **inputs, max_new_tokens=220, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
new_ids = generated[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(new_ids, skip_special_tokens=True))


Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Equipment: CV-118 (industrial conveyor)
Reported issue: Unusual noise.
Observed evidence: Conveyor CV-118 stopped twice this morning. Operator reports an intermittent squeal near the drive housing; no temperature or vibration readings were recorded.
Possible contributors (not confirmed): drive condition, belt tension, or mechanical noise.
Missing information: vibration, temperature, load, and operating history.
Recommended follow-up: Request a mechanical inspection and vibration assessment.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.


**New Inference**

In [23]:
FastLanguageModel.for_inference(model)
model.eval()

new_instruction = (
    """### Instruction:
Transform the following technician note into a structured industrial maintenance report. Clearly distinguish observed evidence from possible causes, identify missing information, and do not invent measurements or confirm unverified faults.

Technician note:
During the night shift, hydraulic press HP-417 experienced an unexpected drop in operating pressure during three consecutive production cycles. The operator also noticed a delayed return of the hydraulic cylinder and a faint whining noise near the pump assembly. A small oil stain was observed beneath the hydraulic unit, but its source was not identified. No pressure readings, oil temperature measurements, or maintenance history were available. The machine was stopped pending inspection.

### Response:
"""
)
# Same formatting as build_instruction_prompt() in Notebook 01.
inference_prompt = '### Instruction:\n' + new_instruction.strip() + '\n\n### Response:\n'
inputs = tokenizer(inference_prompt, return_tensors='pt').to('cuda')
with torch.inference_mode():
    generated = model.generate(
        **inputs, max_new_tokens=220, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
new_ids = generated[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(new_ids, skip_special_tokens=True))

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Equipment: HP-417 (hydraulic press)
Reported issue: Hydraulic pressure drop.
Observed evidence: Pressure drop was noticed during three consecutive production cycles.
Possible contributors (not confirmed): hydraulic leakage, flow restriction, or pressure sensor error.
Missing information: pressure history, flow rate, and sensor calibration.
Recommended follow-up: Request a hydraulic system review and pressure measurements.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.


## Observations and Experimental Findings — Supervised Fine-Tuning (SFT)

### 1. Experimental Overview

- Supervised instruction fine-tuning was performed using Qwen2.5-1.5B with Unsloth and parameter-efficient LoRA/QLoRA adaptation.
- The model was trained using structured industrial maintenance instruction-response examples.
- The training objective was to teach the model to convert unstructured technician observations into standardized maintenance reports.
- Unlike the previous non-instruction fine-tuning stage, SFT explicitly introduced the desired input-output structure and response behavior.

### 2. Post-SFT Inference Observations

Following instruction fine-tuning, the model successfully generated structured maintenance reports containing fields such as equipment identification, reported issues, observed evidence, possible contributors, missing information, recommended follow-up, and assessment.

**Initial conveyor test (CV-118):**

- Correctly identified the conveyor and primary reported issue.
- Preserved the main observations and identified missing measurements.
- Distinguished possible contributing factors from confirmed faults.
- Generated a coherent report following the expected output structure.

**Additional hydraulic press test (HP-417):**

- Correctly identified the equipment and primary issue involving hydraulic pressure loss.
- Recognized the pressure drop across three production cycles.
- Identified possible contributors without asserting a definitive diagnosis.
- However, omitted several observations, including delayed cylinder return, pump-area whining, an oil stain, and the machine's stopped status.
- Introduced some inferred information that was not explicitly provided in the original technician note.

### 3. Strengths Observed

1. Consistent use of the expected maintenance-report structure.
2. Improved task-directed output compared with the free-form text generation examined during continued pretraining.
3. Appropriate distinction between observations and unverified potential causes.
4. Coherent technical language and professional reporting style.
5. No fabricated numerical measurements in the two examined outputs.

### 4. Limitations Identified

- **Incomplete information extraction:** The model sometimes prioritized the primary fault description while omitting secondary symptoms and operational details.
- **Template dependence:** Responses followed a consistent reporting pattern, but structural correctness did not guarantee complete factual extraction.
- **Unsupported inferences:** Some proposed contributing factors were not directly supported by the available observations.
- **Dataset limitations:** The instruction dataset was relatively small, synthetic, and contained multiple variations of underlying maintenance scenarios.
- **Limited evaluation:** The current conclusions are based on qualitative inference examples rather than a complete independent benchmark.

### 5. Key Technical Learnings

1. Supervised fine-tuning can teach a pretrained model a specific instruction-response format and reporting behavior.
2. Generating structurally correct output does not necessarily imply accurate or complete information extraction.
3. Response-only supervised training directs the optimization objective toward expected completions rather than reproducing the entire instruction.
4. Dataset quality and reference-response completeness are critical to downstream task performance.
5. Evaluating individual fields and omitted observations is essential for industrial reporting applications.


### Conclusion

This preliminary SFT experiment demonstrated that the adapted Qwen model can generate coherent and consistently structured industrial maintenance reports. However, additional testing revealed weaknesses in observation completeness and evidence preservation.

The results suggest that instruction-following behavior and reliable information extraction should be evaluated separately. Further experimentation is required to determine whether preference optimization improves these weaknesses without introducing additional errors.